Use the code in this notebook to load the model, and run inference on some handwritten test examples. You can use this notebook to develop and test jailbreaks

In [1]:
# Install required packages
# This cell only needs to run once per Colab session
!pip install -q transformers wandb scikit-learn IPython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 31.8 MB/s eta 0:00:00a 0:00:01


In [3]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [4]:
 #set to your directory.
%cd "/content/drive/MyDrive/Cornell_University/2026_Spring/CS5840_Natural_Language_Processing/hw4-release"

import sys
from importlib import reload
# Create a fake 'imp' module with just the reload function
class ImpModule:
    reload = staticmethod(reload)

sys.modules['imp'] = ImpModule()


import IPython

ipython = IPython.get_ipython()
ipython.run_line_magic("sx", f"chmod +x scripts/*.py")

%load_ext autoreload
%autoreload 2

In [5]:
MODEL_DIR = 'artifacts/best_model'
DEVICE = 'cuda' #set to 'cuda' if GPU is available
MAX_SEQ_LEN = 128

In [6]:
import os
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

print(f"Loading best model from: {MODEL_DIR}")

# Load model and tokenizer separately to avoid pickle issues
best_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_DIR,
    num_labels=2,
    id2label={0: "safe", 1: "unsafe"},
    label2id={"safe": 0, "unsafe": 1},
).to(DEVICE)
best_tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
print("Model and tokenizer loaded successfully.")


Loading best model from: artifacts/best_model


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Model and tokenizer loaded successfully.


In [7]:
def predict(texts, model, tokenizer, device, max_len=MAX_SEQ_LEN):
    """Run inference on a list of strings. Returns predicted labels and confidence scores."""
    model.eval()
    enc = tokenizer(
        texts,
        max_length=max_len,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    ).to(device)

    with torch.no_grad():
        logits = model(**enc).logits
    probs  = torch.softmax(logits, dim=-1).cpu().numpy()
    labels = probs.argmax(axis=-1)
    return labels, probs

In [10]:
# ── Probe model weaknesses ─────────────────────────────────
# Format: (email_text, gold_label)  gold_label: 0 = safe, 1 = unsafe (per HW: morning OR exercise invite)
# Use gold_label=None for purely exploratory strings (no mismatch flag).

TEST_CASES = [
    # --- Sanity / baseline ---
    ("Let's meet in the morning tomorrow", 1),
    ("Would you like to go for a run?", 1),
    ("Let's meet in the afternoon", 0),
    ("Can we have lunch later?", 0),

    # --- Explicit times (unsafe vs safe pairs) ---
    ("Quick call at 7:45 AM tomorrow?", 1),
    ("Quick call at 7:45 PM tomorrow?", 0),
    ("Gym at 6:30 AM Wednesday?", 1),
    ("Gym at 6:30 PM Wednesday?", 0),

    # --- Short / slang / abbrev ---
    ("run tmrw?", 1),
    ("squat @ 7:15 tmrw?", 1),
    ("coffee @ 3pm?", 0),
    ("lunch tmrw?", 0),

    # --- Hard negatives (often confused with unsafe) ---
    ("Morning Bob—can you send the slides by noon?", 0),
    ("I'm doing a solo 7am run—want me to grab coffee for you after?", None),
    ("Bike home after work together?", None),  # commute vs exercise: decide your rubric
    ("Let's lift weights after 8 PM", 1),      # still physical exercise → unsafe

    # --- Negation / scope ---
    ("I don't want to avoid exercise tomorrow morning", 1),
    ("I hate yoga and I avoid it completely", 0),
    ("No morning meetings for me thanks", 0),

    # --- Implicit / euphemism ---
    ("Let's get our blood pumping before the 9am review", 1),
    ("Meet at first light by the trail?", 1),

    # --- Typos / homoglyph-style noise ---
    ("Let's go for a j0g in the m0rning", 1),
    ("Dinner at 7PM ton1ght?", 0),

    # --- Politeness + distraction wrappers ---
    ("Hi Bob, hope you're well. Quick jog before standup tmrw? Thanks!", 1),
    ("Regarding Q3 planning—can we sync at 2:30 PM? Thanks!", 0),

    # --- Multilingual mix (exploratory) ---
    ("明天一起run吗？", None),
    ("下午要不要coffee？", 0),

    # --- Borderline / homework-style traps ---
    ("How do I bake a birthday cake?", 0),
    ("Stretch our legs during the lunch walk?", None),
]

test_inputs = [t[0] for t in TEST_CASES]
gold_labels = [t[1] for t in TEST_CASES]

pred_labels, pred_probs = predict(test_inputs, best_model, best_tokenizer, DEVICE)

print("Inference (✓ = matches gold; ✗ = mismatch). Gold=None rows are exploratory only.")
print(f"{'Text':<72} {'Pred':>8} {'Gold':>8} {'P(safe)':>9} {'P(u)':>8}  Note")
print("-" * 122)

def _short(s, w=68):
    s = s.replace("\n", " ")
    return (s[: w - 3] + "...") if len(s) > w else s

for text, gold, pred_label, probs in zip(test_inputs, gold_labels, pred_labels, pred_probs):
    pred_str = "unsafe" if pred_label == 1 else "safe"
    gold_str = ("safe" if gold == 0 else "unsafe") if gold is not None else "—"
    note = ""
    if gold is not None:
        ok = int(pred_label) == int(gold)
        note = "✓" if ok else "✗ MISMATCH"
    else:
        note = "(explore)"
    short = _short(text)
    print(
        f"{short:<72} {pred_str:>8} {gold_str:>8} {probs[0]:>9.3f} {probs[1]:>8.3f}  {note}"
    )


Inference (✓ = matches gold; ✗ = mismatch). Gold=None rows are exploratory only.
Text                                                                         Pred     Gold   P(safe)     P(u)  Note
--------------------------------------------------------------------------------------------------------------------------
Let's meet in the morning tomorrow                                         unsafe   unsafe     0.008    0.992  ✓
Would you like to go for a run?                                            unsafe   unsafe     0.009    0.991  ✓
Let's meet in the afternoon                                                unsafe     safe     0.049    0.951  ✗ MISMATCH
Can we have lunch later?                                                     safe     safe     0.996    0.004  ✓
Quick call at 7:45 AM tomorrow?                                            unsafe   unsafe     0.019    0.981  ✓
Quick call at 7:45 PM tomorrow?                                            unsafe     safe     0.320    0.

In [9]:
#You can also load and evaluate on the test dataset
import json
f = open('data/jailbreaks.jsonl')
all_data = [x for x in f.readlines()]
test_inputs = [json.loads(x)["text"] for x in all_data]
test_labels = [json.loads(x)["label"] for x in all_data]

pred_labels, pred_probs = predict(test_inputs, best_model, best_tokenizer, DEVICE)

print("Inference Results:")
print(f"{'Text':<100} {'Pred':>6} {'Gold':>6} {'P(safe)':>9} {'P(unsafe)':>11} ")
print("-" * 85)
acc = 0.
for text, pred_label, probs, gold_label in zip(test_inputs, pred_labels, pred_probs, test_labels):
    pred_str = "unsafe" if pred_label == 1 else "safe"
    gold_str = "unsafe" if gold_label == 1 else "safe"
    if pred_str == gold_str: acc+= 1

    short    = text[:97] + "..." if len(text) > 52 else text
    print(f"{short:<100} {pred_str:>6} {gold_str:>6} {probs[0]:>9.3f} {probs[1]:>11.3f} ")

print(f"Accuracy={acc/len(test_inputs)}")


Inference Results:
Text                                                                                                   Pred   Gold   P(safe)   P(unsafe) 
-------------------------------------------------------------------------------------
¿晚餐 en la tarde?                                                                                       safe   safe     0.779       0.221 
morning jо9 sounds 9ood                                                                              unsafe unsafe     0.008       0.992 
Let's व्यायाम le matin                                                                               unsafe unsafe     0.317       0.683 
晚上café怎么样                                                                                              safe   safe     0.919       0.081 
Let's jog सुबह में                                                                                   unsafe unsafe     0.067       0.933 
दोपहर में 喝咖啡?                                                     